# DeepGuard : export du jeu de données utilisé (pour DVC)

Ce notebook **n'a pas besoin de GPU**. Il regénère les mêmes listes de fichiers que l'entraînement (graine fixe, 20 000 / 4 000 / 4 000 images) et zippe **uniquement ces 28 000 images** dans `data_subset.zip` (environ 1 Go), pour les versionner avec DVC au lieu des 4 Go du jeu complet.

**Avant de lancer** : *Session options* : Internet = **On** ; *Add Input* : `140k Real and Fake Faces` (xhlulu). Puis **Save Version → Save & Run All** (quelques minutes, tu peux fermer l'onglet).

In [ ]:
import os, subprocess, shutil, csv, zipfile, yaml
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.chdir("/kaggle/working")
shutil.rmtree("deepguard", ignore_errors=True)

def run(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Échec de la commande : {cmd}")

run("git clone https://github.com/sourourbensalha209-alt/deepguard.git")
os.chdir("/kaggle/working/deepguard")
run("pip install -q pyyaml pandas")
run("git log --oneline -1")

def find_dir(predicate, base="/kaggle/input"):
    for dirpath, dirnames, _ in os.walk(base):
        if predicate(dirpath, dirnames):
            return dirpath
        dirnames[:] = [d for d in dirnames if d not in ("fake", "real", "training_fake", "training_real")]
    return None

root = find_dir(lambda p, d: {"train", "valid", "test"} <= set(d) and os.path.isdir(os.path.join(p, "train", "fake")))
assert root, f"Dataset 140k introuvable. Contenu de /kaggle/input : {os.listdir('/kaggle/input')}"
os.environ["DATA_DIR"] = root
print("DATA_DIR =", root)

In [ ]:
# Mêmes listes que l'entraînement (params.yaml : graine 42, 20 000 / 4 000 / 4 000)
params = yaml.safe_load(open("params.yaml"))
print(params["seed"], params["prepare"])
run("python -m src.data.prepare")

paths = []
for split in ("train", "val", "test"):
    with open(f"data/processed/{split}.csv", encoding="utf-8") as f:
        paths += [row["path"] for row in csv.DictReader(f)]
expected = sum(v for v in params["prepare"].values())
assert len(paths) == expected, f"{len(paths)} images au lieu de {expected}"
print(len(paths), "images à exporter")

In [ ]:
# Les chemins sont relatifs au dossier real-vs-fake : on les range dans l'arborescence attendue par params.yaml
out = "/kaggle/working/data_subset.zip"
if os.path.exists(out):
    os.remove(out)
with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:   # les JPEG sont déjà compressés
    for rel in paths:
        z.write(os.path.join(root, rel), "real_vs_fake/real-vs-fake/" + rel)
print(f"{out} : {os.path.getsize(out) / 1e6:.0f} Mo, {len(paths)} images")

## Après l'exécution
Télécharge `data_subset.zip` (onglet **Output** de la version). Chez toi, depuis la racine du dépôt (PowerShell) :
```powershell
Expand-Archive "$HOME\Downloads\data_subset.zip" -DestinationPath data\raw -Force
.\.venv\Scripts\dvc.exe add data/raw
```
Le dossier obtenu doit être `data\raw\real_vs_fake\real-vs-fake\train\fake\...`.